[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/antfolk/BMEN35_2026/blob/main/Session3/BMEN35_Ex3_01_MLP_from_scratch_step_by_step.ipynb)

# A neural network from the inside: a small MLP

Let's build a small multilayer perceptron (MLP) using NumPy. We will first look at a curved two-class dataset. Then we will calculate a forward pass, work through the loss and gradients, update the weights, and finally draw the decision boundary.

The data are made with `make_moons`. They are coordinates. The curved pattern is useful here because one straight line cannot separate the two groups very well.

This is a lot of code, so run one cell at a time. Stop at the plots and check that you understand what each step changed.

## 1. Imports and create the two-moons data

Each row in `X` is an observation with two features. `y` contains the class number. We use a fixed random seed so the same points appear each time.

We split the data into training, validation, and test sets. The network learns from the training set. We use the validation set to follow training, and keep the test set for a final evaluation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay, classification_report

rng = np.random.default_rng(0)

# Generate two curved groups with a little overlap.
X, y = make_moons(n_samples=500, noise=0.25, random_state=0)

# First reserve a test set for the final evaluation.
X_development, X_test, y_development, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Split the remaining data into training and validation sets.
X_train, X_validation, y_train, y_validation = train_test_split(
    X_development, y_development, test_size=0.2, random_state=43, stratify=y_development
)

print('Training observations:', len(y_train))
print('Validation observations:', len(y_validation))
print('Test observations:', len(y_test))
print('Feature shape:', X_train.shape)

## 2. Look at the data

Each point is one observation. The two colors represent the two classes. The pattern is curved, so one straight line cannot perfectly divide the classes.

This visualization gives us a concrete question for the network: can it learn a curved decision boundary from only two inputs?

In [ ]:
for class_number, color, label in [(0, 'red', 'Class 0'), (1, 'blue', 'Class 1')]:
    rows = y_train == class_number
    plt.scatter(X_train[rows, 0], X_train[rows, 1], c=color,
                label=label, edgecolor='black', alpha=0.75)

plt.xlabel('feature 1')
plt.ylabel('feature 2')
plt.title('Training data: two curved classes')
plt.legend()
plt.grid(alpha=0.25)
plt.show()

### Stop and look: what kind of boundary would fit?

1. Why would one straight line leave many points on the wrong side?
2. Sketch mentally a boundary that could separate the two curved groups.
3. What might happen if the data had more noise or the classes overlapped more?

## 3. Set the network size and initialize its parameters

The network has two input features, one hidden layer with 16 neurons, and two output scores (one for each class).

The weight matrices connect neighboring layers. Biases let each neuron shift its activation. We initialize weights to small random values and biases to zero. Different random starts can lead to slightly different learned models.

In [ ]:
input_size = X_train.shape[1]  # two measured coordinates
hidden_size = 16                # number of neurons in the hidden layer
number_of_classes = 2           # the two moon labels

# Randomly initialized weights; the scale keeps early activations moderate.
W1 = rng.normal(0, np.sqrt(2 / input_size), size=(input_size, hidden_size))
b1 = np.zeros((1, hidden_size))
W2 = rng.normal(0, np.sqrt(2 / hidden_size), size=(hidden_size, number_of_classes))
b2 = np.zeros((1, number_of_classes))

print('W1 shape (input to hidden):', W1.shape)
print('b1 shape:', b1.shape)
print('W2 shape (hidden to output):', W2.shape)
print('b2 shape:', b2.shape)

## 4. The forward pass

For a batch of observations, the network calculates:

1. a weighted sum plus bias for each hidden neuron;
2. a ReLU activation, which replaces negative values with zero;
3. output scores for the two classes;
4. softmax probabilities that sum to one for each observation.

We write each step separately so the intermediate arrays are visible.

For one batch, the same operations can be written as matrix equations. Here, each row is one observation.

$$Z^{(1)} = XW^{(1)} + b^{(1)}, \qquad A^{(1)} = \operatorname{ReLU}(Z^{(1)})$$

$$S = A^{(1)}W^{(2)} + b^{(2)}, \qquad P_{ij} = \frac{\exp(S_{ij})}{\sum_k \exp(S_{ik})}.$$

For example, if the batch has $N$ observations, the shapes are $X: N\times2$, $W^{(1)}: 2\times16$, $A^{(1)}: N\times16$, and $P: N\times2$. The matrix multiplication dimensions explain why the code uses `@`.

Before calculating `np.exp`, we subtract the largest class score in each row. This is a numerical-stability step: if a score is very large, its exponential can overflow. Subtracting the same value from every class score does not change the softmax probabilities, because the common factor cancels between the numerator and denominator. The largest shifted score is 0, so its exponential is 1. For example, scores `[1000, 1001]` become `[-1, 0]`, with the same softmax probabilities.

`axis=1` means “find the maximum across the class columns for each observation.” `keepdims=True` keeps the result as a column with shape `(number of observations, 1)`, so NumPy can subtract it from every class score in that row.

In [ ]:
# First layer: combine the input features using weights and biases.
hidden_scores = X_train @ W1 + b1

# ReLU activation: negative hidden scores become zero.
hidden_activations = np.maximum(0, hidden_scores)

# Output layer: calculate one score for each of the two classes.
class_scores = hidden_activations @ W2 + b2

# Softmax: subtract the largest score first for numerical stability.
# Subtract each observation’s largest class score before exponentiating.
# This keeps exp() values at or below 1 and prevents overflow.
# axis=1 selects a maximum per row; keepdims=True keeps shape (N, 1) for broadcasting.
shifted_scores = class_scores - np.max(class_scores, axis=1, keepdims=True)
exp_scores = np.exp(shifted_scores)
probabilities = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)

print('First observation class scores:', np.round(class_scores[0], 3))
print('First observation probabilities:', np.round(probabilities[0], 3))
print('Probability sum:', probabilities[0].sum())

## 5. Cross-entropy loss

For each observation, cross-entropy loss looks at the probability assigned to the true class. Assigning a high probability to the correct class gives a small loss; assigning a low probability gives a larger loss.

The next code cell calculates the average loss for the training data. At the random initialization, we expect the model to be uncertain and the loss to be relatively high.

For $N$ observations, the average loss is

$$L = -\frac{1}{N}\sum_{i=1}^{N} \log(P_{i,y_i}).$$

Here $P_{i,y_i}$ is the probability the model assigned to the correct class for observation $i$.

In [ ]:
number_of_training_rows = len(y_train)
row_numbers = np.arange(number_of_training_rows)

# Select the probability assigned to the correct class for each row.
probability_of_true_class = probabilities[row_numbers, y_train]

# A tiny value inside the logarithm avoids log(0) if a probability is extremely small.
initial_loss = -np.log(probability_of_true_class + 1e-12).mean()
print('Initial training cross-entropy loss:', round(initial_loss, 3))

## 6. Calculate the output-layer gradient

Backpropagation uses the difference between predicted probabilities and the true labels to calculate how each parameter contributed to the loss.

For softmax with cross-entropy, the score gradient is `probability − one-hot target`. We copy the probabilities before modifying them, so the original array remains unchanged.

For softmax probabilities $P$ and one-hot labels $Y$, the gradient of the average loss with respect to the output scores is

$$\frac{\partial L}{\partial S} = \frac{P-Y}{N}.$$

The code starts with `probabilities.copy()` and subtracts one at the true class position. That creates $P-Y$ before dividing by the number of observations.

In [ ]:
# Begin with the gradient of loss with respect to the class scores.
score_gradient = probabilities.copy()

# Subtract one from the score for the true class in every row.
score_gradient[row_numbers, y_train] -= 1

# Average gradients over all training observations.
score_gradient /= number_of_training_rows

# Gradients for output weights and output biases.
W2_gradient = hidden_activations.T @ score_gradient
b2_gradient = np.sum(score_gradient, axis=0, keepdims=True)

print('Output weight gradient shape:', W2_gradient.shape)
print('Output bias gradient shape:', b2_gradient.shape)

## 7. Send the gradient through ReLU to the first layer

The output-layer gradient tells us how the hidden activations should change. We send it backward through the output weights and then through ReLU.

ReLU has gradient zero where its input was negative or zero, because those values were clipped to zero. For positive inputs its gradient is one.

The remaining matrix gradients follow from the chain rule:

$$\frac{\partial L}{\partial W^{(2)}} = (A^{(1)})^T \frac{\partial L}{\partial S}, \qquad \frac{\partial L}{\partial W^{(1)}} = X^T \frac{\partial L}{\partial Z^{(1)}}.$$

The code calculates these one at a time so you can compare each expression with its corresponding matrix multiplication.

In [ ]:
# Move the gradient backward through the output weights.
hidden_gradient = score_gradient @ W2.T

# ReLU blocks gradients for neurons that had non-positive input scores.
hidden_gradient[hidden_scores <= 0] = 0

# Gradients for input-to-hidden weights and hidden biases.
W1_gradient = X_train.T @ hidden_gradient
b1_gradient = np.sum(hidden_gradient, axis=0, keepdims=True)

print('Input weight gradient shape:', W1_gradient.shape)
print('Hidden bias gradient shape:', b1_gradient.shape)

## 8. Update the parameters once

Gradient descent changes each parameter in the direction that lowers the loss. The learning rate controls the size of each change.

We first take one step and recalculate the loss. If the gradient is wired correctly and the learning rate is sensible, the loss should usually decrease after this small update.

In [ ]:
learning_rate = 0.1

# Move each parameter against its gradient.
W1 = W1 - learning_rate * W1_gradient
b1 = b1 - learning_rate * b1_gradient
W2 = W2 - learning_rate * W2_gradient
b2 = b2 - learning_rate * b2_gradient

# Repeat the forward pass to calculate the loss after one update.
hidden_scores = X_train @ W1 + b1
hidden_activations = np.maximum(0, hidden_scores)
class_scores = hidden_activations @ W2 + b2
# Subtract each observation’s largest class score before exponentiating.
# This keeps exp() values at or below 1 and prevents overflow.
# axis=1 selects a maximum per row; keepdims=True keeps shape (N, 1) for broadcasting.
shifted_scores = class_scores - np.max(class_scores, axis=1, keepdims=True)
probabilities = np.exp(shifted_scores) / np.exp(shifted_scores).sum(axis=1, keepdims=True)
loss_after_one_update = -np.log(probabilities[row_numbers, y_train] + 1e-12).mean()

print('Loss after one update:', round(loss_after_one_update, 3))

## 9. Train for multiple iterations

The previous cells showed one update in detail. Training repeats the same forward pass, loss calculation, backpropagation, and parameter update many times.

We place the repeated calculation in a function so the main training loop stays readable. This helper returns the loss and all gradients; the loop applies the update.

In [ ]:
def calculate_loss_and_gradients(features, labels, W1, b1, W2, b2):
    # Forward pass through the hidden layer.
    hidden_scores = features @ W1 + b1
    hidden_activations = np.maximum(0, hidden_scores)

    # Forward pass through the output layer and stable softmax.
    class_scores = hidden_activations @ W2 + b2
    # Subtract each observation’s largest class score before exponentiating.
    # This keeps exp() values at or below 1 and prevents overflow.
    # axis=1 selects a maximum per row; keepdims=True keeps shape (N, 1) for broadcasting.
    shifted_scores = class_scores - np.max(class_scores, axis=1, keepdims=True)
    exp_scores = np.exp(shifted_scores)
    probabilities = exp_scores / exp_scores.sum(axis=1, keepdims=True)

    # Cross-entropy loss for the correct class in each row.
    rows = np.arange(len(labels))
    loss = -np.log(probabilities[rows, labels] + 1e-12).mean()

    # Backpropagate the loss through the output layer.
    score_gradient = probabilities.copy()
    score_gradient[rows, labels] -= 1
    score_gradient /= len(labels)
    W2_gradient = hidden_activations.T @ score_gradient
    b2_gradient = score_gradient.sum(axis=0, keepdims=True)

    # Backpropagate through ReLU and the hidden layer.
    hidden_gradient = score_gradient @ W2.T
    hidden_gradient[hidden_scores <= 0] = 0
    W1_gradient = features.T @ hidden_gradient
    b1_gradient = hidden_gradient.sum(axis=0, keepdims=True)

    return loss, (W1_gradient, b1_gradient, W2_gradient, b2_gradient)


training_iterations = 2500
learning_rate = 0.1
training_losses = []
validation_losses = []

for iteration in range(training_iterations):
    # Calculate loss and gradients using training observations only.
    loss, gradients = calculate_loss_and_gradients(X_train, y_train, W1, b1, W2, b2)
    W1_gradient, b1_gradient, W2_gradient, b2_gradient = gradients

    # Apply one gradient-descent update to all weights and biases.
    W1 -= learning_rate * W1_gradient
    b1 -= learning_rate * b1_gradient
    W2 -= learning_rate * W2_gradient
    b2 -= learning_rate * b2_gradient

    # Save training loss at every iteration for a learning curve.
    training_losses.append(loss)

    # Calculate validation loss periodically to check generalization during training.
    if iteration % 25 == 0:
        validation_loss, _ = calculate_loss_and_gradients(X_validation, y_validation, W1, b1, W2, b2)
        validation_losses.append(validation_loss)

print('First recorded training loss:', round(training_losses[0], 3))
print('Last recorded training loss:', round(training_losses[-1], 3))

## 10. Plot training and validation loss

Training loss shows how closely the network fits its training data. Validation loss gives a check on observations that do not update the weights. If training loss keeps falling while validation loss rises, the network may be overfitting. The separate test set remains untouched until the final evaluation.

In [ ]:
plt.plot(training_losses, label='training loss')
validation_iterations = np.arange(0, training_iterations, 25)
plt.plot(validation_iterations[:len(validation_losses)], validation_losses,
         label='validation loss', linewidth=2)
plt.xlabel('iteration')
plt.ylabel('cross-entropy loss')
plt.title('Loss while training the small MLP')
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 11. Evaluate the test observations

We use the final weights to predict test labels and report accuracy and the confusion matrix. The per-class report shows whether one class is harder to recognize than the other.

In [ ]:
def predict_classes(features, W1, b1, W2, b2):
    # Run the forward pass using the learned parameters.
    hidden = np.maximum(0, features @ W1 + b1)
    scores = hidden @ W2 + b2

    # The class with the largest score is the predicted class.
    return np.argmax(scores, axis=1)

test_predictions = predict_classes(X_test, W1, b1, W2, b2)
print('Test accuracy:', round(accuracy_score(y_test, test_predictions), 3))
print(classification_report(y_test, test_predictions, target_names=['Class 0', 'Class 1']))

ConfusionMatrixDisplay.from_predictions(y_test, test_predictions, cmap='Blues')
plt.title('MLP test-set confusion matrix')
plt.show()

## 12. Plot the learned decision boundary

We create a grid of locations across the two-feature plot and ask the network to classify each location. The colored background shows the decision regions; the dots show the training observations.

This lets us compare the learned boundary with the curved shape we saw before fitting.

In [ ]:
# Make a mesh that covers the training feature space.
x_min, x_max = X_train[:, 0].min() - 0.5, X_train[:, 0].max() + 0.5
y_min, y_max = X_train[:, 1].min() - 0.5, X_train[:, 1].max() + 0.5
grid_x, grid_y = np.meshgrid(
    np.linspace(x_min, x_max, 250),
    np.linspace(y_min, y_max, 250)
)
# ravel() turns each 2D grid into a 1D list of coordinates.
grid_points = np.c_[grid_x.ravel(), grid_y.ravel()]
# Put the flat predictions back into the 2D grid shape for contourf.
grid_classes = predict_classes(grid_points, W1, b1, W2, b2).reshape(grid_x.shape)

plt.contourf(grid_x, grid_y, grid_classes, alpha=0.25, cmap='coolwarm')
for class_number, color, label in [(0, 'tomato', 'Class 0'), (1, 'royalblue', 'Class 1')]:
    rows = y_train == class_number
    plt.scatter(X_train[rows, 0], X_train[rows, 1], c=color,
                label=label, edgecolor='black', alpha=0.75)
plt.xlabel('feature 1')
plt.ylabel('feature 2')
plt.title('Decision boundary learned by the MLP')
plt.legend()
plt.show()

## 13. Specific conclusions

1. Which calculation turns hidden-layer scores into activations? What effect does it have on negative values?
2. What does the cross-entropy loss reward the network for doing?
3. Did training and validation loss follow the same pattern? What might a widening gap indicate?
4. Does the plotted boundary follow the curved training data? Point to one place where the boundary still makes mistakes.
5. What did we gain by writing forward and backward passes explicitly, and what would we gain by using a neural-network library?

The next notebook builds a CNN for image data. A convolutional layer uses small learned filters across an image, while a dense layer connects all inputs to all neurons.